# recs_028 — Stage 4 LLM ranker spike

Foundational-slice spike: does a local LLM reranker (`ranker_llm_local.py`, Ablation C arm 1 —
minimal ranked-list prompt, no reasoning) beat the shipped v2a heuristic ranker?

- **Frozen cohort:** `val_llm_mini_v1` (200 examples, a deterministic subset of `val_dev_12k_v1`,
  not an independent resample — see `docs/plans/rag_stage4_llm_ranker_plan.md`).
- **Pool:** `two_tower_v1` @100, from `artifacts/recs/offline_eval/runs/llm_mini/eval_offline_examples.jsonl`
  (built with `include_query_text_in_examples_jsonl: true`).
- **Fair baseline:** v2a (`two_tower_v1_v2a_embed_query_logpop_blend`) recomputed on this *same*
  200-example cohort — not the 12k-cohort 0.095/0.070 number, which isn't comparable at this sample size.
- **Backend:** `LlamaCppBackend`, `Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf`, CUDA (RTX 5070/Blackwell).

Status: **spike**, not wired to `pool_rerank_registry`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from steam_review_ml.evaluation.candidate_text import build_candidate_text_lookup
from steam_review_ml.evaluation.example_cohort import load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import pool_rerank_registry, rerank_scores_on_pool
from steam_review_ml.evaluation.retrieval_offline_eval import (
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)
from steam_review_ml.recommender.llm_backends import LlamaCppBackend
from steam_review_ml.recommender.ranker_llm_local import precompute_llm_local_scores_by_ex_idx

REPO_ROOT = Path.cwd().parent.parent
K_FINAL = 10

POOLS_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/llm_mini/eval_offline_examples.jsonl"
GGUF_PATH = REPO_ROOT / "artifacts/models/llm_local/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

pool_rows = load_retrieval_pools_jsonl(POOLS_JSONL, method="two_tower_v1")
print(f"{len(pool_rows)} pool rows (two_tower_v1, k_retrieval={pool_rows[0]['retrieval_k']})")

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-24 09:04:48.606664: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 09:04:48.637946: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787576688.653210 1221684 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787576688.657998 1221684 cuda_bl

200 pool rows (two_tower_v1, k_retrieval=100)


## Build query text + candidate text lookups

`query_text` comes from the pool jsonl directly (opt-in field). Candidate text is built once,
over the union of every app_id appearing in any of the 200 pools -- not per-example, to avoid
redundantly repeating the same IGDB join across ~100 candidates x 200 examples.

In [2]:
import json

query_text_by_ex_idx = {int(r["ex_idx"]): r["query_text"] for r in pool_rows}

all_pool_app_ids: set[int] = set()
for r in pool_rows:
    all_pool_app_ids.update(json.loads(r["retrieved_app_ids_json"]))

candidate_texts = build_candidate_text_lookup(all_pool_app_ids)
print(f"{len(candidate_texts)} unique candidate app_ids across all pools")

315 unique candidate app_ids across all pools


In [3]:
candidate_texts

{688130: 'Pogostuck: Rage With Your Friends\n\nClimb a surreal mountain on a pogo stick and make friends along the way.',
 626690: 'Sword Art Online: Fatal Bullet\n\nSword Art Online: Fatal Bullet is a Third Person Shooter Role-Playing Game (TPSRPG), scheduled to be released on February 8, 2018 on the PlayStation 4, XBox One, as well as Personal Computers (PCs) via Steam. The game is being developed by Dimps, based on Unreal Engine 4, and will be set in Gun Gale Online. It will be the first Sword Art Online TPSRPG and the first Sword Art Online game on the XBox One platform.\n\nThe game will follow the events of Sword Art Online: Hollow Realization. Gun Gale Online was released by Zaskar, following the advent of The Seed. The Death Gun incident will take place in the game, but the story will be different from the canon version. There will be some new events and players can obtain an element that has not appeared in the canon universe.',
 264200: 'One Finger Death Punch\n\nExperience ci

## Load the backend and calibrate timing

Time one real full-size (@100 candidates) call before committing to the full 200-example run --
the earlier smoke test only used truncated 10-candidate pools, so this is a different cost.

In [4]:
import time

backend = LlamaCppBackend(str(GGUF_PATH), n_gpu_layers=-1)

sample_row = pool_rows[0]
sample_ex_idx = int(sample_row["ex_idx"])
print(f"Sample example index: {sample_ex_idx}, query text: {query_text_by_ex_idx[sample_ex_idx]}")
sample_pool_apps = [int(x) for x in json.loads(sample_row["retrieved_app_ids_json"])]
sample_candidates = [
    {"app_id": aid, "text": candidate_texts.get(aid, "")} for aid in sample_pool_apps
]

t0 = time.time()
_ = backend.generate_ranking(query_text_by_ex_idx[sample_ex_idx], sample_candidates, top_k=K_FINAL)
elapsed = time.time() - t0
print(f"One full @{len(sample_pool_apps)}-candidate call: {elapsed:.1f}s")
print(f"Estimated total for {len(pool_rows)} examples: {elapsed * len(pool_rows) / 60:.1f} min")

Sample example index: 0, query text: My squad ransacked a Slave outpost and freed all the slaves, and found out that a slave lord was living in a house inside of the walls. So we then killed his bodyguards, took all his loot and carried him back alive to our base and put him in our prisoner cage to slowly watch him starve. We then placed him in a peeler machine and watched as his limbs fell off. We then healed him up and fed him some food to prevent him from dying. He's now in the middle of the town, unguarded and occasionally fed. 

I guess the moral of the story is that being a slave lord is worse than being a slave in Kenshi when my squad is around.

(Also, this was a modded file!)
One full @100-candidate call: 4.7s
Estimated total for 200 examples: 15.7 min


## Run the full precompute (all 200 examples)

Asks the backend for its **top-10 picks** from each ~100-candidate pool, not a full ranking
of every candidate -- eval only looks at the top `K_FINAL` anyway, and asking a local 7-8B
model to produce a complete, valid permutation of a ~100-item pool in one shot turned out to
be unreliable in practice (0/3 real examples succeeded in early testing). Asking for only the
top 10 fixed that (4/5 succeeded in a follow-up check) -- a far more tractable structured-output
task. At ~6s/example this should take roughly `200 * 6 / 60 ≈ 20` minutes.

In [5]:
t0 = time.time()
llm_scores_by_ex_idx, llm_failures_by_ex_idx = precompute_llm_local_scores_by_ex_idx(
    backend,
    pool_rows,
    query_text_by_ex_idx=query_text_by_ex_idx,
    candidate_texts=candidate_texts,
    top_k=K_FINAL,
    verbose=True,
)
elapsed = time.time() - t0
n_examples = len(pool_rows)
n_failures = len(llm_failures_by_ex_idx)
print(f"\nDone in {elapsed/60:.1f} min.")
print(f"Parse-failure rate: {n_failures}/{n_examples} = {n_failures/n_examples:.1%}")

  LLM local precompute 10/200...
  LLM local precompute 20/200...
  LLM local precompute 30/200...
  LLM local precompute 40/200...
  LLM local precompute 50/200...
  LLM local precompute 60/200...
  LLM local precompute 70/200...
  LLM local precompute 80/200...
  LLM local precompute 90/200...
  LLM local precompute 100/200...
  LLM local precompute 110/200...
  LLM local precompute 120/200...
  LLM local precompute 130/200...
  LLM local precompute 140/200...
  LLM local precompute 150/200...
  LLM local precompute 160/200...
  LLM local precompute 170/200...
  LLM local precompute 180/200...
  LLM local precompute 190/200...

Done in 16.6 min.
Parse-failure rate: 28/200 = 14.0%


In [14]:
llm_scores_by_ex_idx[0]

array([-0.78164576, -0.78164576,  4.        ,  5.        , -0.78164576,
        8.        ,  9.        , -0.78164576, -0.78164576,  7.        ,
        6.        , -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, 10.        , -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576,  2.        , -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576,  3.        , -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78164576,
       -0.78164576, -0.78164576, -0.78164576, -0.78164576, -0.78

## Compute metrics for both rankers

Same low-level metric functions `retrieval_offline_eval.py` uses everywhere else
(`hit_rate_at_k`, `ndcg_at_k`, `mrr`), applied directly within each pool: since a pool is
already a fixed subset of the catalog, `pool_app_ids` itself doubles as the "catalog" for
this call, and `argsort(-scores)` gives ranked positions into it -- no need for the
full-catalog `app_to_row` indirection those functions are more commonly used with.

In [6]:
def metrics_for_pool(pool_app_ids: list[int], scores: np.ndarray, positives: set[int], k_final: int) -> dict:
    pool_arr = np.asarray(pool_app_ids)
    order = np.argsort(-np.asarray(scores))[:k_final]
    return {
        "Hit@K": hit_rate_at_k(order, positives, k_final, pool_arr),
        "NDCG@K": ndcg_at_k(order, positives, k_final, pool_arr),
        "MRR": mrr(order, positives, pool_arr),
    }


def metrics_table(pools: list[dict], scores_by_ex_idx: dict[int, np.ndarray], *, k_final: int) -> pd.DataFrame:
    rows = []
    for row in pools:
        ex_idx = int(row["ex_idx"])
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_app_ids = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        rows.append({"ex_idx": ex_idx, **metrics_for_pool(pool_app_ids, scores_by_ex_idx[ex_idx], positives, k_final)})
    return pd.DataFrame(rows)


llm_metrics_df = metrics_table(pool_rows, llm_scores_by_ex_idx, k_final=K_FINAL)
llm_metrics_df[["Hit@K", "NDCG@K", "MRR"]].mean()

Hit@K     0.065000
NDCG@K    0.033206
MRR       0.023708
dtype: float64

## Recompute v2a on the same cohort (fair baseline)

The shipped v2a NDCG@10 (0.095 overall / 0.070 Slice A) is measured on the full 12k-cohort --
not comparable to a 200-example run. Recomputing it here, on this exact `val_llm_mini_v1`
cohort, via the same `pool_rerank_registry()` spec used in production, gives an apples-to-apples
baseline.

In [7]:
V2A_METHOD = "two_tower_v1_v2a_embed_query_logpop_blend"

catalog_ctx = load_ranking_catalog_context(repo_root=REPO_ROOT)
v2a_spec = pool_rerank_registry()[V2A_METHOD]

v2a_scores_by_ex_idx: dict[int, np.ndarray] = {}
for row in pool_rows:
    ex_idx = int(row["ex_idx"])
    pool_app_ids = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    retr_scores = json.loads(row["retrieved_scores_json"])
    v2a_scores_by_ex_idx[ex_idx] = rerank_scores_on_pool(
        pool_app_ids,
        retr_scores,
        v2a_spec,
        pop_row=catalog_ctx.pop_row,
        app_to_row=catalog_ctx.app_to_row,
        query_app_id=int(row["query_app_id"]),
    )

v2a_metrics_df = metrics_table(pool_rows, v2a_scores_by_ex_idx, k_final=K_FINAL)
v2a_metrics_df[["Hit@K", "NDCG@K", "MRR"]].mean()

Hit@K     0.205000
NDCG@K    0.096996
MRR       0.068234
dtype: float64

## Comparison

Both methods on the identical 200-example cohort. Promotion bar (for reference, from the
12k-cohort): v2a must be beaten on both overall and Slice A NDCG@10 -- not directly applicable
at n=200, but the sign and gap here is the real signal from this spike.

In [8]:
summary = pd.DataFrame(
    {
        "ranker_llm_local": llm_metrics_df[["Hit@K", "NDCG@K", "MRR"]].mean(),
        "v2a (same cohort)": v2a_metrics_df[["Hit@K", "NDCG@K", "MRR"]].mean(),
    }
).T
summary["n_examples"] = [len(llm_metrics_df), len(v2a_metrics_df)]
summary["parse_failure_rate"] = [n_failures / n_examples, 0.0]
summary

,Hit@K,NDCG@K,MRR,n_examples,parse_failure_rate
ranker_llm_local,0.065,0.033206,0.023708,200,0.14
v2a (same cohort),0.205,0.096996,0.068234,200,0.00


In [9]:
sample_candidates

[{'app_id': 875210,
  'text': '三国群英传8 Heroes of the Three Kingdoms 8\n\nHeroes of the Three Kingdoms 8 is the eighth installment in the Heroes of the Three Kingdoms series of strategic war games based on the classic Chinese novel Romance of the Three Kingdoms. The first main-line sequel in over 13 years, the game gives the series a new look while staying true to its core formula.'},
 {'app_id': 792990,
  'text': "Identity\n\nIdentity is a new breed of massively multiplayer online role-playing game where hundreds of players interact in a world of absolute freedom. It's the actions of players which determine your fate and the fate of the world you live in."},
 {'app_id': 394360,
  'text': 'Hearts of Iron IV\n\nVictory is at your fingertips! Your ability to lead your nation is your supreme weapon, the strategy game Hearts of Iron IV lets you take command of any nation in World War II; the most engaging conflict in world history.\n\nFrom the heart of the battlefield to the command center, 